<a href="https://colab.research.google.com/github/Rokiafaysal/advanced-rag-chatbot/blob/main/rag_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf faiss-cpu sentence-transformers groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 16.0 MB/s eta 0:00:00


In [2]:
import pypdf
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq

In [31]:
from google.colab import userdata
client = Groq(api_key=userdata.get('Groq_key'))


In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
from pypdf import PdfReader
reader= PdfReader("/content/drive/MyDrive/front_matter.pdf")

In [34]:
text = "\n".join(p.extract_text() for p in reader.pages)


In [35]:
model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
def split_text(text, chunk_size=500,overlap=50):
  words = text.split()
  chunks=[]
  for i in range(0,len(words),chunk_size - overlap):
    chunk=" ".join(words[i:i+chunk_size])
    chunks.append(chunk)
  return chunks




In [37]:

chunks = split_text(text)
print(f" num of chunks: {len(chunks)}")

 num of chunks: 52


In [38]:
embedding=model.encode(chunks)
index=faiss.IndexFlatL2(embedding.shape[1])
index.add(embedding)
print({index.ntotal})

{52}


In [39]:
def search_query(query , k=3):
  query_embedding= model.encode([query])
  distances, indices = index.search(query_embedding, k)
  return [chunks[i] for i in indices[0]  ]


In [40]:
results = search_query("what is deep learning")
for r in results:
    print(r)
    print("---")

D. J., Koh, P. W., and Ng, A. (2010). Tiled convolutional neural networks. In J. Laﬀerty, C. K. I. Williams, J. Shawe-Taylor, R. Zemel, and A. Culotta, editors,Advances in Neural Information Processing Systems 23 (NIPS’10), pages 1279–1287. 346 Le, Q., Ngiam, J., Coates, A., Lahiri, A., Prochnow, B., andNg, A.(2011). Onoptimization methods for deep learning. InProc. ICML’2011. ACM. 312 Le, Q., Ranzato, M., Monga, R., Devin, M., Corrado, G., Chen, K., Dean, J., and Ng, A. (2012). Building high-level features using large scale unsupervised learning. In ICML’2012. 22, 23 Le Roux, N. and Bengio, Y. (2008). Representational power of restricted Boltzmann machines and deep belief networks.Neural Computation, 20(6), 1631–1649. 551, 652 Le Roux, N. and Bengio, Y. (2010). Deep belief networks are compact universal approxi- mators. Neural Computation, 22(8), 2192–2207. 551 LeCun, Y. (1985). Une procédure d’apprentissage pour Réseau à seuil assymétrique. In Cognitiva 85: A la Frontière de l’Intell

In [41]:
model_chat="llama-3.3-70b-versatile"

In [42]:
def call_llm(query, chunks, model_chat=model_chat, temperature=0.4, max_tokens=300):
  context = "\n".join(chunks)
  prompt = f"""
You are an assistant helping students understand this book.

context from this book
{context}
    Question: {query}
   answer based only on the context above.

  """

  response = client.chat.completions.create(
    model=model_chat,
    messages=[
      {
        "role": "user",
        "content": prompt
      }
    ],
    temperature=temperature,
    max_tokens=max_tokens
  )

  return response.choices[0].message.content



In [43]:
results = search_query("what is deep learning")
answer = call_llm("what is deep learning", results)
print(answer)

Based on the provided context, deep learning refers to a subfield of machine learning that involves the use of neural networks with multiple layers to learn complex patterns and representations in data. The context mentions various techniques, architectures, and applications of deep learning, such as convolutional neural networks, recurrent neural networks, and deep belief networks, but does not provide a explicit definition.

However, from the references and citations, it can be inferred that deep learning is a field that focuses on the development of algorithms and models that can learn and represent complex data, such as images, speech, and text, using multiple layers of neural networks.

Some key aspects of deep learning that can be gathered from the context include:

* The use of multiple layers of neural networks to learn complex patterns and representations
* The application of deep learning to various domains, such as computer vision, speech recognition, and natural language pr